# LitScope — Step 1: Fetch Papers
Fetch papers from OpenAlex and save them to Google Drive.

**Run order:** Run this notebook first, then run `02_classify_papers.ipynb`

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Change this: path to your data folder in Drive ────────────────────────────
DRIVE_DIR = '/content/drive/MyDrive/LitScope/data'
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/by_platform', exist_ok=True)
print(f'Data directory: {DRIVE_DIR}')

## 2. Install Dependencies

In [ ]:
!pip install requests pandas -q

## 3. Configuration

In [ ]:
# ── Journal configuration ────────────────────────────────────
JOURNALS = [
    {'name': 'Information Systems Research',              'short': 'ISR',  'openalex_id': 'S202812398', 'platform': 'INFORMS / ISR',           'max': 3000},
    {'name': 'MIS Quarterly',                             'short': 'MISQ', 'openalex_id': 'S57293258',  'platform': 'AIS / MISQ',              'max': 2000},
    {'name': 'Journal of Management Information Systems', 'short': 'JMIS', 'openalex_id': 'S9954729',   'platform': 'Taylor & Francis / JMIS', 'max': 2000},
]

# ── Journals to fetch (comment out ones you don't need) ──────
TARGET_JOURNALS = [
    'ISR',
    'MISQ',
    'JMIS',
]

FETCH_BATCH        = 200
OPENALEX_UA        = 'LitScope/1.0 (Academic Research; mailto:researcher@email.com)'
CLASSIFIED_CSV     = f'{DRIVE_DIR}/papers_classified.csv'
RAW_CSV            = f'{DRIVE_DIR}/papers_raw.csv'

CSV_COLUMNS = [
    'title', 'abstract', 'year', 'venue', 'authors',
    'doi', 'url', 'source', 'platform',
    'is_behavioral', 'theories_used', 'confidence', 'reason',
]

print('Config ready')

## 4. Fetch Functions

In [ ]:
import requests
import pandas as pd
import time


def reconstruct_abstract(inverted_index):
    if not inverted_index:
        return ''
    words = {pos: word for word, positions in inverted_index.items() for pos in positions}
    return ' '.join(words[i] for i in sorted(words))


def load_existing(path):
    try:
        df = pd.read_csv(path)
        print(f'Existing data: {len(df)} papers')
        return df
    except FileNotFoundError:
        print('No existing data, starting fresh')
        return pd.DataFrame(columns=CSV_COLUMNS)


def fetch_openalex(journal, exclude_dois):
    papers = []
    url    = 'https://api.openalex.org/works'
    params = {
        'filter':   f"primary_location.source.id:{journal['openalex_id']},has_abstract:true",
        'select':   'title,abstract_inverted_index,publication_year,doi,authorships',
        'per-page': FETCH_BATCH,
        'page':     1,
        'sort':     'publication_year:desc',
    }
    headers = {'User-Agent': OPENALEX_UA}

    print(f"\n[{journal['short']}] Fetching up to {journal['max']} papers")

    while len(papers) < journal['max']:
        try:
            r = requests.get(url, headers=headers, params=params, timeout=20)
            if r.status_code != 200:
                print(f'  HTTP {r.status_code}, stopping')
                break

            items = r.json().get('results', [])
            if not items:
                print('  No more data')
                break

            for item in items:
                abstract = reconstruct_abstract(item.get('abstract_inverted_index') or {})
                if not abstract:
                    continue

                raw_doi  = (item.get('doi') or '').strip()
                doi      = raw_doi.replace('https://doi.org/', '').lower()  # clean identifier only
                if doi in exclude_dois:
                    continue

                authors = [
                    a['author']['display_name']
                    for a in item.get('authorships', []) if a.get('author')
                ]
                papers.append({
                    'title':    item.get('title'),
                    'abstract': abstract,
                    'year':     item.get('publication_year'),
                    'venue':    journal['name'],
                    'authors':  ', '.join(authors),
                    'doi':      doi,                                          # e.g. 10.1287/isre.xxx
                    'url':      f'https://doi.org/{doi}' if doi else '',     # full clickable URL
                    'source':   'OpenAlex',
                    'platform': journal['platform'],
                })
                if doi:
                    exclude_dois.add(doi)

            print(f"  Page {params['page']}, new papers so far: {len(papers)}")
            params['page'] += 1
            time.sleep(1)

        except Exception as e:
            print(f'  Error: {e}')
            time.sleep(5)

    print(f"  [{journal['short']}] Done, {len(papers)} new papers")
    return papers


print('Functions defined')

## 5. Start Fetching and Saving

In [ ]:
# Load existing data and get known DOIs for deduplication
df_existing  = load_existing(CLASSIFIED_CSV)
exclude_dois = set(df_existing['doi'].dropna().str.strip().str.lower())

# Fetch target journals
targets  = [j for j in JOURNALS if j['short'] in TARGET_JOURNALS]
all_new  = []

for journal in targets:
    papers = fetch_openalex(journal, exclude_dois)
    all_new.extend(papers)

print(f'\nTotal new papers: {len(all_new)}')

In [ ]:
if all_new:
    df_new = pd.DataFrame(all_new)

    # Save raw fetch results
    df_new.to_csv(RAW_CSV, index=False, encoding='utf-8-sig')
    print(f'Raw data saved: {RAW_CSV}')

    # Merge into the main data file (is_behavioral left empty, pending classification)
    for col in CSV_COLUMNS:
        if col not in df_new.columns:
            df_new[col] = None

    df_all = pd.concat([df_existing, df_new[CSV_COLUMNS]], ignore_index=True)
    df_all.drop_duplicates(subset=['doi'], keep='first', inplace=True)
    df_all['year'] = pd.to_numeric(df_all['year'], errors='coerce')
    df_all.sort_values('year', ascending=False, inplace=True, na_position='last')
    df_all.reset_index(drop=True, inplace=True)
    df_all.to_csv(CLASSIFIED_CSV, index=False, encoding='utf-8-sig')

    print(f'Main data saved: {CLASSIFIED_CSV}')
    print(f'Total: {len(df_all)} papers')
    print(f'Pending classification: {df_all["is_behavioral"].isna().sum()} papers')
    print(f'\nNext step: run 02_classify_papers.ipynb')
else:
    print('No new papers, nothing to save')